# 04 Statistical Analysis

We are going to this notebook for deeper analysis such as correlation checks, hypothesis testing, forecasting, segmentation, or regression.

In [1]:
from pathlib import Path

import pandas as pd
from scipy import stats

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().resolve().name == 'notebooks' else Path.cwd().resolve()

In [2]:
DATA_PATH = 'https://raw.githubusercontent.com/pushkar-bit/Section-E_G-19_Healthcare-Patient-Readmission-Intelligence/main/data/processed/cleaned_hospital_readmissions.csv'
df = pd.read_csv(DATA_PATH)
df.head()

,age,time_in_hospital,n_lab_procedures,n_procedures,n_medications,n_outpatient,n_inpatient,n_emergency,medical_specialty,diag_1,...,total_visits,is_frequent_patient,stay_intensity,procedures_per_day,meds_per_visit,service_utilization_score,length_of_stay_bucket,age_numeric,age_group,is_senior
0,3,8,72,1,18,2,0,0,4,0,...,2,0,576,0,9,34.5,0,75.0,3,1
1,3,3,34,2,13,0,0,0,5,6,...,0,0,102,1,13,18.1,2,75.0,3,1
2,1,5,45,0,18,0,0,0,4,0,...,0,0,225,0,18,23.4,1,55.0,2,0
3,3,2,36,0,12,1,0,0,4,0,...,1,0,72,0,12,18.0,2,75.0,3,1
4,2,1,42,0,7,0,0,0,3,6,...,0,0,42,0,7,18.9,2,65.0,3,1


##**1. Correlation Analysis**

In [3]:
corr = df.corr(numeric_only=True)["readmitted_30_days"].drop("readmitted_30_days")

corr_df = (
    corr.abs()
    .sort_values(ascending=False)
    .head(8)
    .reset_index()
)

corr_df.columns = ["Feature", "Correlation"]

display(corr_df)

print(f"Insight: '{corr_df.iloc[0]['Feature']}' has the strongest correlation with readmission.")

,Feature,Correlation
0,n_inpatient,0.212480
1,total_visits,0.207957
2,is_frequent_patient,0.180341
3,meds_per_visit,0.096812
4,n_outpatient,0.095487
5,n_emergency,0.093519
6,diabetes_med,0.062145
7,procedures_per_day,0.060936


Insight: 'n_inpatient' has the strongest correlation with readmission.


##**2. Logistic Regression**

In [8]:
# --- Logistic Regression with Clinical + Operational Drivers ---

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
import pandas as pd

# Select features
# Clinical + Operational proxies
features = [
    "n_inpatient",              # clinical
    "total_visits",             # clinical
    "time_in_hospital",         # operational proxy (length of stay)
    "n_medications",            # clinical
    "medical_specialty"         # operational (department)
]

# One-hot encode categorical variable (medical_specialty)
X = pd.get_dummies(df[features], columns=["medical_specialty"], drop_first=True)

y = df["readmitted_30_days"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# Coefficients
coef_df = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

# Sort by importance
coef_df["Abs_Impact"] = coef_df["Coefficient"].abs()
coef_df = coef_df.sort_values("Abs_Impact", ascending=False)

display(coef_df.drop(columns="Abs_Impact").head(10))

# --- Insight ---
top_feature = coef_df.iloc[0]["Feature"]

print(f"Insight: '{top_feature}' has the strongest influence on readmission probability.")
print("Operational drivers (length of stay, department) show measurable impact alongside clinical factors.")

,Feature,Coefficient
9,medical_specialty_6,-0.272445
0,n_inpatient,0.250329
8,medical_specialty_5,-0.242324
1,total_visits,0.144162
6,medical_specialty_3,-0.121007
5,medical_specialty_2,0.054078
4,medical_specialty_1,0.048440
7,medical_specialty_4,0.021096
2,time_in_hospital,0.019083
3,n_medications,0.003576


Insight: 'medical_specialty_6' has the strongest influence on readmission probability.
Operational drivers (length of stay, department) show measurable impact alongside clinical factors.


##**3. Model Performance**

In [5]:
y_pred = model.predict(X_test)

print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.60      0.82      0.69      2658
           1       0.65      0.38      0.48      2342

    accuracy                           0.61      5000
   macro avg       0.62      0.60      0.58      5000
weighted avg       0.62      0.61      0.59      5000



##**4. Hypothesis Testing — Chi-square**

In [6]:
from scipy import stats

cat_vars = [
    ("age_group", "Age"),
    ("medical_specialty", "Specialty"),
    ("length_of_stay_bucket", "LOS"),
]

results = []

for col, name in cat_vars:
    table = pd.crosstab(df[col], df["readmitted_30_days"])
    chi2, p, _, _ = stats.chi2_contingency(table)

    results.append({
        "Variable": name,
        "p-value": round(p, 5),
        "Significant": "Yes" if p < 0.05 else "No"
    })

chi2_df = pd.DataFrame(results)
display(chi2_df)

print("Insight: All key categorical variables show statistically significant association with readmission.")

,Variable,p-value,Significant
0,Age,0.0,Yes
1,Specialty,0.0,Yes
2,LOS,0.0,Yes


Insight: All key categorical variables show statistically significant association with readmission.


##**Summary - Table**

In [9]:
summary_stats = [
    {
        "Method": "Correlation",
        "Finding": f"{corr_df.iloc[0]['Feature']} has highest correlation",
        "Interpretation": "Strongest linear relationship with readmission"
    },
    {
        "Method": "Regression",
        "Finding": f"{coef_df.iloc[0]['Feature']} has highest impact",
        "Interpretation": "Most influential predictor in probability model"
    },
    {
        "Method": "Chi-square",
        "Finding": "Categorical variables show low p-values",
        "Interpretation": "Statistically significant association with readmission"
    },
    {
        "Method": "Operational Drivers",
        "Finding": "Length of stay & department show measurable impact",
        "Interpretation": "Operational factors influence readmission risk"
    }
]

summary_df = pd.DataFrame(summary_stats)
display(summary_df)

,Method,Finding,Interpretation
0,Correlation,n_inpatient has highest correlation,Strongest linear relationship with readmission
1,Regression,medical_specialty_6 has highest impact,Most influential predictor in probability model
2,Chi-square,Categorical variables show low p-values,Statistically significant association with rea...
3,Operational Drivers,Length of stay & department show measurable im...,Operational factors influence readmission risk
